### [ 전처리 ]


In [16]:
## -------------------------------------
## 이미지 데이터 로딩 & 기본 정보
## -------------------------------------
import cv2
import os
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import numpy as np
import matplotlib.cm as cm
import sys
sys.path.append(r"C:\KDT14\7_CV")

import importlib
import cv_utils

importlib.reload(cv_utils)

<module 'cv_utils' from 'C:\\KDT14\\7_CV\\cv_utils.py'>

**[0] 사전준비**
- 모듈로딩
- 폴더 체크 및 생성

In [12]:

## --------------------------------------------------
## 전역 변수 및 상수
## --------------------------------------------------
## => 이미지 관련

ROI_DIR  = '../Data/Roi'

## => 수집 데이터
DATA_LABEL= ['apple', 'banana']
APPLE_ROI= f'{ROI_DIR}/apple'
BANANA_ROI= f'{ROI_DIR}/banana'


# 경로 확인
print(ROI_DIR, APPLE_ROI, BANANA_ROI, sep='\n')

../Data/Roi
../Data/Roi/apple
../Data/Roi/banana


In [3]:
## 폴더 생성 및 체크
## --------------------------------------------------
os.makedirs(APPLE_ROI, exist_ok=True)
os.makedirs(BANANA_ROI, exist_ok=True)

In [13]:
## => 이미지 크기 | 50% 줄인 크기 확인
DATA_DIRS = [APPLE_ROI, BANANA_ROI]

for target in DATA_DIRS:
    filelist = os.listdir(target)

    for filename in filelist:
        IMG_PATH = f'{target}/{filename}'
        SAVE_PATH = f'{target}/re_{filename}'

        print(f'IMG_PATH : {IMG_PATH}, SAVE_PATH: {SAVE_PATH}')

        imgNP = cv2.imread(IMG_PATH, cv2.IMREAD_GRAYSCALE)

        ## 비율 유지하면서 50% 크기 줄이기
        resizeNP= cv2.resize(imgNP, (0,0), fx=0.5, fy=0.5)
        resizeNP= cv2.resize(resizeNP, (100, 100))

        ## 이미지 파일 저장
        is_save= cv2.imwrite(SAVE_PATH, resizeNP)



IMG_PATH : ../Data/Roi/apple/apple_0000_crop.jpg, SAVE_PATH: ../Data/Roi/apple/re_apple_0000_crop.jpg
IMG_PATH : ../Data/Roi/apple/apple_0001_crop.jpg, SAVE_PATH: ../Data/Roi/apple/re_apple_0001_crop.jpg
IMG_PATH : ../Data/Roi/apple/apple_0002_crop.jpg, SAVE_PATH: ../Data/Roi/apple/re_apple_0002_crop.jpg
IMG_PATH : ../Data/Roi/apple/apple_0003_crop.jpg, SAVE_PATH: ../Data/Roi/apple/re_apple_0003_crop.jpg
IMG_PATH : ../Data/Roi/apple/apple_0005_crop.jpg, SAVE_PATH: ../Data/Roi/apple/re_apple_0005_crop.jpg
IMG_PATH : ../Data/Roi/apple/apple_0008_crop.jpg, SAVE_PATH: ../Data/Roi/apple/re_apple_0008_crop.jpg
IMG_PATH : ../Data/Roi/apple/apple_0009_crop.jpg, SAVE_PATH: ../Data/Roi/apple/re_apple_0009_crop.jpg
IMG_PATH : ../Data/Roi/apple/apple_0011_crop.jpg, SAVE_PATH: ../Data/Roi/apple/re_apple_0011_crop.jpg
IMG_PATH : ../Data/Roi/apple/apple_0012_crop.jpg, SAVE_PATH: ../Data/Roi/apple/re_apple_0012_crop.jpg
IMG_PATH : ../Data/Roi/apple/apple_0014_crop.jpg, SAVE_PATH: ../Data/Roi/apple/re_

**[2] 데이터셋 생성**
- DataFrame 으로 저장
- DataFrame=> csv 형식 변환 저장
- 형식
    * 픽셀, 픽셀, ..., 픽셀, 라벨
    * 라벨, 파일명, 픽셀, ..., 픽셀

- 픽셀 데이터: cv2.imread() 2D/ 3D==> 1D
- 라벨 데이터: 폴더명==> 경로 추출 및 하드 코딩 지정

In [14]:
## => 파일 폴더와 라벨 설정
DATA_DIRS = [APPLE_ROI, BANANA_ROI]
LABELS = ['apple', 'banana']          ## APPLE_ROI.split('/')[-1]

## => 이미지 파일의 로우 데이터 저장
data_list = []

for target, label in zip(DATA_DIRS, LABELS):

    filelist = os.listdir(target)

    for filename in filelist:
        if not filename.lower().endswith(
            ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
        ):
            continue

        IMG_PATH = f'{target}/{filename}'
        imgNP = cv2.imread(IMG_PATH, cv2.IMREAD_GRAYSCALE)



    flatten_img= imgNP.flatten()

    row= [label, filename]+ flatten_img.tolist()+ [label]
    data_list.append(row)


In [18]:
import os
import cv2
import pandas as pd

# 반드시 초기화
data_list = []

DATA_DIRS = [APPLE_ROI, BANANA_ROI]
LABELS = ["apple", "banana"]

EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

for target, label in zip(DATA_DIRS, LABELS):

    filelist = os.listdir(target)

    for filename in filelist:

        if not filename.lower().endswith(EXTS):
            continue

        IMG_PATH = os.path.join(str(target), filename)

        imgNP = cv2.imread(
            IMG_PATH,
            cv2.IMREAD_GRAYSCALE
        )

        if imgNP is None:
            print(f"이미지 읽기 실패: {IMG_PATH}")
            continue

        # 모든 이미지가 224×224인지 확인
        if imgNP.shape != (224, 224):
            print(f"크기 다름: {filename}, 현재 크기={imgNP.shape}")
            continue

        flatten_img = imgNP.flatten()

        # 파일명 + 픽셀 50,176개 + 라벨
        row = [filename] + flatten_img.tolist() + [label]

        data_list.append(row)


# =====================================================
# CSV 저장
# =====================================================

# filename + pixel 50,176개 + label
columns = (
    ["filename"]
    + [f"pixel{i}" for i in range(224 * 224)]
    + ["label"]
)

SAVE_CSV_PATH = "fruit_data.csv"

print("행 데이터 길이:", len(data_list[0]))
print("컬럼 길이:", len(columns))

df = pd.DataFrame(
    data_list,
    columns=columns
)

df.to_csv(
    SAVE_CSV_PATH,
    index=False
)

print("데이터프레임 크기:", df.shape)
print(f"저장 완료: {SAVE_CSV_PATH}")

크기 다름: re_apple_0000_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0001_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0002_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0003_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0005_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0008_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0009_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0011_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0012_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0014_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0015_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0016_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0017_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0018_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0019_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0020_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0021_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0022_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0024_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0025_crop.jpg, 현재 크기=(100, 100)
크기 다름: re_apple_0026_crop.jpg, 현재 크기=(10

In [17]:
## ------------------------------------------------------------
## => CSV 파일로 저장
## ------------------------------------------------------------
## 컬럼명 만들기: pixel0, ~ ,pixel9999, label
columns = [f'pixel{i}' for i in range(100*100)] + ['label']

## 파일명
SAVE_CSV_PATH = 'fruit_data.csv'

## => DataFrame으로 저장
df = pd.DataFrame(data_list, columns=columns)
df.to_csv(SAVE_CSV_PATH, index=False)

print(df.shape)
print(f'저장 완료: {SAVE_CSV_PATH}')

ValueError: 10001 columns passed, passed data had 10003 columns